# Ranga — Synthetic Dataset Generator

Builds **function-calling** training data for Gemma 4 12B. Hospital and insurance data is never baked into model answers — the model learns to call the same five tools the Flutter app uses (`hospital_navigation_tool.dart`).

## Outputs (`ranga_output/`)

| File | Records | Purpose |
|---|---|---|
| `ranga_sft_500.jsonl` | 500 | Multi-turn tool trajectories (SFT) |
| `ranga_dpo_200.jsonl` | 200 | Chosen vs rejected trajectories (DPO) |
| `ranga_eval_50.jsonl` | 50 | Queries + expected tool-call sequences |
| `ranga_tools.json` | 1 | Tool schemas (shared across all records) |

## Fixed tool pipeline

Two call paths — the model must always end with `rankHospitalsByPriorityAndCost`:

**Nearby path** (general routing):
```
getCurrentLocation → getInsuranceCoverageBlock → getNearbyHospitals → rankHospitalsByPriorityAndCost → answer
```

**Condition path** (symptom / specialty routing):
```
getCurrentLocation → getInsuranceCoverageBlock → searchHospitalsByCondition → rankHospitalsByPriorityAndCost → answer
```

## Source data

| Layer | File | Role |
|---|---|---|
| Hospitals (SQLite sim) | `health_facilities.geojson` | Provider records near ALU |
| Insurance rules | `InPatient-Medical-Insurance-Contract.md` | Britam policy prose |
| Insurance rules | `Old_Mutual_Insurance_Rwanda_Medical_Insurance_Tariff.md` | UAP / Old Mutual tariff |
| Query seeds | [MT Samples](https://huggingface.co/datasets/harishnair04/mtsamples) | De-identified complaint snippets |

## 0 — Install dependencies (Google Colab)

1. **Runtime → Change runtime type → T4 GPU** (or better).
2. Add a Colab secret named `HF_TOKEN` with your [Hugging Face token](https://huggingface.co/settings/tokens) — required for gated Gemma weights.
3. Run the cell below once, then **Runtime → Restart session** and run from section 1.

In [1]:
!pip install -U "transformers>=4.52" accelerate bitsandbytes datasets pandas tqdm ipywidgets --quiet

## 1 — Config

In [2]:
import json, math, random, re, pathlib, textwrap, copy
from datetime import datetime

DATASET_DIR   = pathlib.Path(".")
GEOJSON_PATH  = DATASET_DIR / "health_facilities.geojson"
BRITAM_MD     = DATASET_DIR / "InPatient-Medical-Insurance-Contract.md"
OLD_MUTUAL_MD = DATASET_DIR / "Old_Mutual_Insurance_Rwanda_Medical_Insurance_Tariff.md"

OUT_DIR    = DATASET_DIR / "ranga_output"
OUT_DIR.mkdir(exist_ok=True)
SFT_PATH   = OUT_DIR / "ranga_sft_500.jsonl"
DPO_PATH   = OUT_DIR / "ranga_dpo_200.jsonl"
EVAL_PATH  = OUT_DIR / "ranga_eval_50.jsonl"
TOOLS_PATH = OUT_DIR / "ranga_tools.json"

MODEL_ID    = "google/gemma-4-12B-it"
MAX_TOKENS  = 512
TEMPERATURE = 0.7
SEED        = 42

ALU_LAT, ALU_LON = -1.9695, 30.1589
MAX_RADIUS_KM    = 25
TOP_PROVIDERS    = 15

N_TEMPLATE = 300
N_CUSTOM   = 200
N_DPO      = 200
N_EVAL     = 50

# Mirrors HospitalNavigationTool pipelines in app/lib/services/hospital_navigation_tool.dart
TOOL_PIPELINE_NEARBY    = ["getCurrentLocation", "getInsuranceCoverageBlock", "getNearbyHospitals", "rankHospitalsByPriorityAndCost"]
TOOL_PIPELINE_CONDITION = ["getCurrentLocation", "getInsuranceCoverageBlock", "searchHospitalsByCondition", "rankHospitalsByPriorityAndCost"]

random.seed(SEED)
print(f"Output: {OUT_DIR.resolve()}")

Output: /Users/x/Documents/capstone/dataset/ranga_output


## 2 — Load Gemma 4 12B (CUDA)

Used **only** to generate natural-language user queries (section 7b). All tool calls and tool results are deterministic.

Loads on GPU via Hugging Face Transformers. Uses 4-bit quantization automatically on GPUs with ≤20 GB VRAM (e.g. Colab T4).

In [ ]:
import os
import torch
from huggingface_hub import login
from transformers import AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

hf_token = os.environ.get("HF_TOKEN")
if IN_COLAB and not hf_token:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)
else:
    print("Tip: set HF_TOKEN (Colab secret or env var) for faster Gemma downloads.")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} ({props.total_memory / 1e9:.1f} GB VRAM)")
else:
    raise RuntimeError("No CUDA GPU found — enable GPU in Colab: Runtime → Change runtime type → T4 GPU.")

print(f"Loading {MODEL_ID} …")
processor = AutoProcessor.from_pretrained(MODEL_ID, token=hf_token)

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
if vram_gb <= 20:
    load_kwargs = {
        "quantization_config": BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_quant_type="nf4",
        ),
        "device_map": "auto",
    }
    print("Using 4-bit quantization (fits Colab T4).")
else:
    load_kwargs = {"torch_dtype": torch.bfloat16, "device_map": "auto"}
    print("Using bfloat16 full precision.")

model = AutoModelForMultimodalLM.from_pretrained(MODEL_ID, token=hf_token, **load_kwargs)
model.eval()
print("Model ready.")

def llm(prompt: str, max_tokens: int = MAX_TOKENS, temp: float = TEMPERATURE) -> str:
    messages = [{"role": "user", "content": prompt}]
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True,
        enable_thinking=False,
    ).to(model.device)
    input_len = inputs["input_ids"].shape[-1]
    gen_kwargs = {"max_new_tokens": max_tokens}
    if temp > 0:
        gen_kwargs.update(do_sample=True, temperature=temp, top_p=0.95)
    with torch.inference_mode():
        outputs = model.generate(**inputs, **gen_kwargs)
    response = processor.decode(outputs[0][input_len:], skip_special_tokens=True)
    return response.strip()

/Users/x/Documents/capstone/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading mlx-community/gemma-4-12B-it-4bit …


Fetching 9 files:  33%|███▎      | 3/9 [01:45<03:31, 35.31s/it]


## 3 — Complaint seeds (MT Samples)

In [ ]:
from datasets import load_dataset
import pandas as pd

raw = load_dataset("harishnair04/mtsamples", split="train")
df  = raw.to_pandas()
print(f"Loaded {len(df)} rows")
df.head(2)

In [ ]:
SPECIALTY_MAP = {
    "Allergy / Immunology": "General Consultation", "Cardiovascular / Pulmonary": "General Consultation",
    "Dentistry": "Dental Care", "Dermatology": "General Consultation",
    "Emergency Room Reports": "Emergency Services", "ENT - Otolaryngology": "General Consultation",
    "Gastroenterology": "Laboratory Tests", "General Medicine": "General Consultation",
    "Hematology - Oncology": "Laboratory Tests", "Neurology": "General Consultation",
    "Obstetrics / Gynecology": "General Consultation", "Ophthalmology": "Optical Care",
    "Orthopedic": "Physical Therapy", "Pain Management": "Physical Therapy",
    "Pathology": "Laboratory Tests", "Pediatrics - Neonatal": "General Consultation",
    "Physical Medicine - Rehab": "Physical Therapy", "Podiatry": "General Consultation",
    "Psychiatry / Psychology": "General Consultation", "Radiology": "Laboratory Tests",
    "Rheumatology": "General Consultation", "Sleep Medicine": "General Consultation",
    "Speech - Language": "Physical Therapy", "Urology": "General Consultation",
}
DISCARD = {"Autopsy", "Neurosurgery", "Surgery", "Bariatrics"}
CLINICAL_BLOCKLIST = re.compile(
    r'\b(diagnosis|diagnos\w+|prescri\w+|medic\w+|drug|tablet|capsule|mg|ml|dose|'
    r'injection|iv |surgery|surgical|procedure|biopsy|catheter|resect\w+|'
    r'transplant|bypass|suture|incision|abscess|carcinoma|malignant|'
    r'hypertension|diabetes|atrial|fibrillation|edema|sepsis)\b', re.IGNORECASE)
CRISIS_KEYWORDS = re.compile(r'\b(suicid\w+|self.harm|kill myself|want to die|hopeless)\b', re.IGNORECASE)

def extract_context(row) -> dict | None:
    specialty = str(row.get("medical_specialty", "")).strip()
    if specialty in DISCARD or specialty not in SPECIALTY_MAP:
        return None
    desc = str(row.get("description", "")).strip()
    if len(desc) < 20 or CRISIS_KEYWORDS.search(desc):
        return None
    clean_desc = CLINICAL_BLOCKLIST.sub("", desc).strip()
    clean_desc = re.sub(r'\s{2,}', ' ', clean_desc)
    transcription = str(row.get("transcription", ""))[:200]
    age_match = re.search(r'(\d+)[- ]year[- ]old', transcription, re.IGNORECASE)
    age_band = "adult"
    if age_match:
        age = int(age_match.group(1))
        age_band = "child" if age < 18 else ("elderly" if age >= 65 else "adult")
    return {
        "service_category": SPECIALTY_MAP[specialty],
        "age_band": age_band,
        "complaint_seed": clean_desc[:200],
        "has_chronic": bool(re.search(r'history of|chronic|recurrent', desc, re.IGNORECASE)),
        "source_row_id": int(row.name),
    }

contexts = [c for _, row in df.iterrows() if (c := extract_context(row)) is not None]
print(f"Usable contexts: {len(contexts)}")

## 4 — Dynamic data layer

Simulates the on-device SQLite database. Updated at runtime via JSON delta sync in production.

### 4a — Hospitals from GeoJSON

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2) -> float:
    R = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = math.radians(lat2-lat1), math.radians(lon2-lon1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2 * R * math.asin(math.sqrt(a))

def feature_centroid(geometry: dict):
    if geometry["type"] == "Point":
        lon, lat = geometry["coordinates"]
        return lon, lat
    if geometry["type"] == "Polygon":
        ring = geometry["coordinates"][0]
        return sum(c[0] for c in ring)/len(ring), sum(c[1] for c in ring)/len(ring)
    return None, None

AMENITY_TO_TIER = {
    "pharmacy": "Tier 1 — Pharmacy", "clinic": "Tier 1 — Health Centre",
    "doctors": "Tier 1 — Health Centre", "hospital": "Tier 2 — District Hospital",
}
TIER3_KEYWORDS = re.compile(r'king faisal|military|district hospital|teaching|oncology|cancer', re.I)
SERVICE_AMENITY = {
    "pharmacy": ["Pharmacy & Medication"],
    "clinic": ["General Consultation", "Laboratory Tests"],
    "doctors": ["General Consultation"],
    "hospital": ["General Consultation", "Dental Care", "Optical Care",
                 "Laboratory Tests", "Emergency Services", "Physical Therapy"],
}
SERVICE_TO_SPECIALTIES = {
    "General Consultation": ["general medicine"],
    "Dental Care": ["dentistry", "dental"],
    "Optical Care": ["ophthalmology"],
    "Laboratory Tests": ["general medicine"],
    "Pharmacy & Medication": ["general medicine"],
    "Emergency Services": ["emergency"],
    "Physical Therapy": ["orthopedics", "general medicine"],
}

with open(GEOJSON_PATH) as f:
    geo = json.load(f)

rows = []
for feat in geo["features"]:
    props = feat["properties"]
    name = (props.get("name") or props.get("name_en") or "").strip()
    if not name: continue
    amenity = props.get("amenity") or props.get("healthcare") or ""
    if amenity not in SERVICE_AMENITY or props.get("adm1_name") != "Kigali City": continue
    lon, lat = feature_centroid(feat["geometry"])
    if lon is None: continue
    dist = haversine_km(ALU_LAT, ALU_LON, lat, lon)
    if dist > MAX_RADIUS_KM: continue
    tier = AMENITY_TO_TIER.get(amenity, "Tier 2 — District Hospital")
    if TIER3_KEYWORDS.search(name): tier = "Tier 3 — Referral Hospital"
    rows.append({
        "provider_id": props["id"], "name": name,
        "physical_address": f"{props.get('adm3_name','')}, {props.get('adm2_name','')}, Kigali",
        "district": props.get("adm3_name", ""), "province": "Kigali City",
        "distance_from_alu_km": round(dist, 2), "phone_number": "",
        "amenity": amenity, "tier_level": tier,
        "latitude": lat, "longitude": lon, "active": 1,
        "type": "private" if "clinic" in name.lower() or "legacy" in name.lower() else "public",
    })

hospitals = (
    pd.DataFrame(rows).sort_values("distance_from_alu_km")
    .drop_duplicates("name").head(TOP_PROVIDERS).reset_index(drop=True)
)
print(f"Providers within {MAX_RADIUS_KM} km: {len(hospitals)}")
hospitals[["name", "tier_level", "distance_from_alu_km"]].head(8)

### 4b — Insurance rules from Markdown

In [ ]:
# Training labels students use → network keys the app stores on hospital records
SCHEME_LABELS = ["CBHI", "RSSB", "Britam", "Old Mutual", "Sanlam", "MMI", "Radiant"]
SCHEME_TO_NETWORK = {
    "CBHI": "mutuelle", "RSSB": "rssb", "Britam": "britam",
    "Old Mutual": "uap", "EdenCare": "uap", "Sanlam": "sanlam",
    "MMI": "mmi", "Radiant": "radiant",
}
OUT_OF_NETWORK_BY_SCHEME = {
    "Britam": {"Legacy Hospital", "Legacy Clinics"},
    "CBHI": set(), "RSSB": set(), "Old Mutual": set(),
    "Sanlam": set(), "MMI": set(), "Radiant": set(),
}
BASE_FEES_RWF = {
    "General Consultation": 15_000, "Dental Care": 25_000, "Optical Care": 30_000,
    "Laboratory Tests": 20_000, "Pharmacy & Medication": 10_000,
    "Emergency Services": 50_000, "Physical Therapy": 20_000,
}
SERVICES = list(BASE_FEES_RWF.keys())

def load_markdown_sources() -> dict[str, str]:
    return {"britam": BRITAM_MD.read_text(), "old_mutual": OLD_MUTUAL_MD.read_text()}

def parse_britam_rules(text: str) -> dict:
    return {"scheme_name": "Britam", "source_file": BRITAM_MD.name,
            "default_copayment_rate": 0.10, "requires_referral_tier3": True}

def parse_old_mutual_rules(text: str) -> dict:
    return {"scheme_name": "Old Mutual", "source_file": OLD_MUTUAL_MD.name,
            "default_copayment_rate": 0.10, "deductible_rwf": 5_000}

PUBLIC_SCHEMES = {
    "CBHI": {"scheme_name": "CBHI", "default_copayment_rate": 0.10, "requires_referral": True},
    "RSSB": {"scheme_name": "RSSB", "default_copayment_rate": 0.15},
}
PRIVATE_SCHEMES = {"Britam": parse_britam_rules(load_markdown_sources()["britam"]),
                   "Old Mutual": parse_old_mutual_rules(load_markdown_sources()["old_mutual"])}

def build_insurance_rules(hospitals_df: pd.DataFrame) -> pd.DataFrame:
    records, rule_id = [], 1
    for _, prov in hospitals_df.iterrows():
        oon_names = set()
        for scheme, oon_set in OUT_OF_NETWORK_BY_SCHEME.items():
            if prov["name"] in oon_set: oon_names.add(scheme)
        tier = prov["tier_level"]
        amenity_services = SERVICE_AMENITY.get(prov["amenity"], SERVICES)
        for scheme, meta in list(PUBLIC_SCHEMES.items()) + list(PRIVATE_SCHEMES.items()):
            net_key = SCHEME_TO_NETWORK[scheme]
            for service in SERVICES:
                if service not in amenity_services and prov["amenity"] != "hospital": continue
                records.append({
                    "rule_id": rule_id, "provider_id": prov["provider_id"],
                    "scheme_name": scheme, "network_key": net_key,
                    "service_category": service, "tier_level": tier,
                    "copayment_rate": meta["default_copayment_rate"],
                    "requires_referral": int("Tier 3" in tier or meta.get("requires_referral", False)),
                    "requires_pre_auth": int(scheme in ("Old Mutual", "Britam") and "Tier 3" in tier),
                    "out_of_network": int(scheme in oon_names),
                    "base_fee_rwf": BASE_FEES_RWF[service],
                    "deductible_rwf": meta.get("deductible_rwf", 0),
                    "active": 1,
                })
                rule_id += 1
    return pd.DataFrame(records)

rules = build_insurance_rules(hospitals)
rules_full = rules.merge(hospitals, on="provider_id", suffixes=("", "_h"))

# Build accepted_insurance per hospital (network keys, matching Flutter SQLite)
accepted_map = {}
for pid, grp in rules_full.groupby("provider_id"):
    keys = grp.loc[grp["out_of_network"]==0, "network_key"].unique().tolist()
    accepted_map[pid] = sorted(set(keys))
hospitals["accepted_insurance"] = hospitals["provider_id"].map(accepted_map)
hospitals["specialties"] = hospitals["amenity"].map(
    lambda a: ["general medicine"] if a != "hospital" else ["general medicine", "emergency"]
)

rules_in_network = rules_full[(rules_full["active"]==1) & (rules_full["out_of_network"]==0)].reset_index(drop=True)
print(f"Rules: {len(rules_full)} total | in-network: {len(rules_in_network)}")
print(f"Schemes: {sorted(rules_full['scheme_name'].unique())}")

## 5 — Ranga tools (fixed)

Schemas and executors mirror `app/lib/services/hospital_navigation_tool.dart` exactly.

### 5a — Tool schemas

In [ ]:
RANGA_TOOLS = [
    {"type": "function", "function": {
        "name": "getCurrentLocation",
        "description": "Get current GPS coordinates (latitude and longitude) of the student.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "getInsuranceCoverageBlock",
        "description": "Retrieve coverage details (copay percent, network key, referral requirements) for the student's insurance plan.",
        "parameters": {"type": "object", "properties": {
            "insurance": {"type": "string", "description": "Insurance plan name as spoken by the student (e.g. CBHI, RSSB, Britam, Old Mutual)"}
        }, "required": ["insurance"]},
    }},
    {"type": "function", "function": {
        "name": "getNearbyHospitals",
        "description": "Retrieve list of nearby hospitals from given coordinates.",
        "parameters": {"type": "object", "properties": {
            "lat": {"type": "number"}, "lng": {"type": "number"},
            "radiusKm": {"type": "number", "default": 25},
            "coverageBlock": {"type": "object", "description": "Coverage block from getInsuranceCoverageBlock"},
        }, "required": ["lat", "lng", "coverageBlock"]},
    }},
    {"type": "function", "function": {
        "name": "searchHospitalsByCondition",
        "description": "Retrieve hospitals matching a medical condition or symptom.",
        "parameters": {"type": "object", "properties": {
            "condition": {"type": "string"},
            "coverageBlock": {"type": "object"},
            "lat": {"type": "number"}, "lng": {"type": "number"},
        }, "required": ["condition", "coverageBlock"]},
    }},
    {"type": "function", "function": {
        "name": "rankHospitalsByPriorityAndCost",
        "description": "Score and rank hospitals by copay, distance, emergency unit, and rating.",
        "parameters": {"type": "object", "properties": {
            "hospitals": {"type": "array", "items": {"type": "object"}},
            "coverageBlock": {"type": "object"},
        }, "required": ["hospitals", "coverageBlock"]},
    }},
]

with open(TOOLS_PATH, "w") as f:
    json.dump(RANGA_TOOLS, f, indent=2)
print(f"Saved {len(RANGA_TOOLS)} tools → {TOOLS_PATH.name}")
for t in RANGA_TOOLS:
    print(f"  • {t['function']['name']}")

### 5b — Runtime executors (SQLite simulator)

In [ ]:
INSURANCE_BLOCKS = {
    "mutuelle": {"providerName": "Mutuelle de Santé (CBHI)", "networkKey": "mutuelle", "copayPercent": 10.0,
        "requiresReferral": True, "referralNotes": "Requires referral letter from a local health center.",
        "coverageDetails": "Covers 90% of medical bills at public health facilities."},
    "rssb": {"providerName": "RSSB / RAMA", "networkKey": "rssb", "copayPercent": 15.0,
        "requiresReferral": False, "referralNotes": "Direct access to certified facilities.",
        "coverageDetails": "Covers 85% of medical bills at partner facilities."},
    "mmi": {"providerName": "Military Medical Insurance (MMI)", "networkKey": "mmi", "copayPercent": 10.0,
        "requiresReferral": False, "referralNotes": "Direct access to Rwanda Military Hospital.",
        "coverageDetails": "Covers 90% for security personnel and dependents."},
    "sanlam": {"providerName": "Sanlam Private Health Insurance", "networkKey": "sanlam", "copayPercent": 15.0,
        "requiresReferral": False, "referralNotes": "Direct access to Legacy Clinics and King Faisal Hospital.",
        "coverageDetails": "Private insurance with direct billing."},
    "britam": {"providerName": "Britam Private Insurance", "networkKey": "britam", "copayPercent": 10.0,
        "requiresReferral": False, "referralNotes": "Direct billing at partner private clinics.",
        "coverageDetails": "Direct coverage for outpatient packages and specialty care."},
    "uap": {"providerName": "UAP Old Mutual", "networkKey": "uap", "copayPercent": 10.0,
        "requiresReferral": False, "referralNotes": "Direct access to premium partner hospitals.",
        "coverageDetails": "Premium private coverage across partner clinics."},
    "radiant": {"providerName": "Radiant Insurance", "networkKey": "radiant", "copayPercent": 15.0,
        "requiresReferral": False, "referralNotes": "Accepted at major public and private providers.",
        "coverageDetails": "Covers standard packages, pharmacy, and emergency visits."},
}

def tool_getCurrentLocation() -> dict:
    return {"lat": ALU_LAT, "lng": ALU_LON}

def tool_getInsuranceCoverageBlock(insurance: str) -> dict:
    clean = insurance.lower().replace(" ", "")
    if "mutuelle" in clean or "cbhi" in clean: return dict(INSURANCE_BLOCKS["mutuelle"])
    if "rssb" in clean or "rama" in clean: return dict(INSURANCE_BLOCKS["rssb"])
    if "mmi" in clean or "military" in clean: return dict(INSURANCE_BLOCKS["mmi"])
    if "sanlam" in clean or "soras" in clean: return dict(INSURANCE_BLOCKS["sanlam"])
    if "britam" in clean: return dict(INSURANCE_BLOCKS["britam"])
    if "uap" in clean or "mutual" in clean or "eden" in clean: return dict(INSURANCE_BLOCKS["uap"])
    if "radiant" in clean: return dict(INSURANCE_BLOCKS["radiant"])
    return {"providerName": insurance or "None", "networkKey": "none", "copayPercent": 100.0,
            "requiresReferral": False, "referralNotes": "Full out-of-pocket payment required.",
            "coverageDetails": "No insurance coverage."}

def _hospital_to_result(prov: pd.Series, dist: float, net_key: str) -> dict:
    accepted = prov.get("accepted_insurance") or []
    net = rules_full[(rules_full["provider_id"]==prov["provider_id"]) & (rules_full["network_key"]==net_key)]
    avg_cost = int(net["base_fee_rwf"].mean()) if not net.empty else 15_000
    return {
        "hospital": {
            "id": prov["provider_id"], "name": prov["name"],
            "address": prov["physical_address"], "lat": prov["latitude"], "lng": prov["longitude"],
            "type": prov.get("type", "public"),
            "acceptedInsurance": accepted,
            "specialties": prov.get("specialties", ["general medicine"]),
            "emergencyUnit": prov["amenity"] == "hospital",
            "openingHours": "24/7", "averageRating": 4.0, "ratingCount": 10,
            "averageCostRwf": avg_cost,
            "averageCostByInsurance": {net_key: avg_cost},
            "costSubmissionCount": 5,
        },
        "distanceKm": round(dist, 2),
        "isInNetwork": net_key in accepted,
    }

def tool_getNearbyHospitals(lat: float, lng: float, radiusKm: float, coverageBlock: dict) -> dict:
    net_key = coverageBlock["networkKey"]
    results = []
    for _, prov in hospitals.iterrows():
        dist = haversine_km(lat, lng, prov["latitude"], prov["longitude"])
        if radiusKm <= 0 or dist <= radiusKm:
            results.append(_hospital_to_result(prov, dist, net_key))
    results.sort(key=lambda x: (not x["isInNetwork"], x["distanceKm"]))
    return {"results": results}

def tool_searchHospitalsByCondition(condition: str, coverageBlock: dict,
                                    lat: float | None = None, lng: float | None = None) -> dict:
    net_key = coverageBlock["networkKey"]
    ref_lat, ref_lng = lat or ALU_LAT, lng or ALU_LON
    cond = condition.lower()
    target_specs = ["general medicine"]
    if any(k in cond for k in ("dent", "teeth", "tooth")): target_specs = ["dentistry", "dental"]
    elif any(k in cond for k in ("eye", "optic", "vision")): target_specs = ["ophthalmology"]
    elif any(k in cond for k in ("emerg", "accident", "injur")): target_specs = ["emergency"]
    elif any(k in cond for k in ("bone", "phys", "therap", "joint")): target_specs = ["orthopedics"]
    results = []
    for _, prov in hospitals.iterrows():
        specs = prov.get("specialties") or ["general medicine"]
        if any(s in specs for s in target_specs) or "general medicine" in specs:
            dist = haversine_km(ref_lat, ref_lng, prov["latitude"], prov["longitude"])
            results.append(_hospital_to_result(prov, dist, net_key))
    results.sort(key=lambda x: (not x["isInNetwork"], x["distanceKm"]))
    return {"results": results}

def tool_rankHospitalsByPriorityAndCost(hospitals_list: list, coverageBlock: dict) -> dict:
    net_key, copay_pct = coverageBlock["networkKey"], coverageBlock["copayPercent"]
    ranked = []
    for res in hospitals_list:
        h, dist = res["hospital"], res["distanceKm"]
        in_net = res["isInNetwork"]
        eff_copay = copay_pct if in_net else 100.0
        baseline = h["averageCostByInsurance"].get(net_key, h["averageCostRwf"]) or 15_000
        est = int(baseline * eff_copay / 100)
        copay_score = max(0.0, min(50.0, 50.0 * (1.0 - est / 30_000)))
        dist_score  = max(0.0, min(30.0, 30.0 * (1.0 - dist / 15.0)))
        bonus = (10.0 if h["emergencyUnit"] else 0) + (20.0 if in_net else 0)
        rating_score = h["averageRating"] * 2.0
        total = copay_score + dist_score + bonus + rating_score
        ranked.append({
            "result": res, "score": total, "estimatedCopayRwf": est,
            "scoreExplanation": f"Score: {total:.1f}/110. Est. Copay: {est} RWF ({eff_copay:.0f}% of {baseline} RWF)",
        })
    ranked.sort(key=lambda x: x["score"], reverse=True)
    return {"rankedResults": ranked}

TOOL_DISPATCH = {
    "getCurrentLocation": lambda **kw: tool_getCurrentLocation(),
    "getInsuranceCoverageBlock": tool_getInsuranceCoverageBlock,
    "getNearbyHospitals": tool_getNearbyHospitals,
    "searchHospitalsByCondition": tool_searchHospitalsByCondition,
    "rankHospitalsByPriorityAndCost": tool_rankHospitalsByPriorityAndCost,
}

def execute_tool(name: str, arguments: dict) -> dict:
    return TOOL_DISPATCH[name](**arguments)

# Sanity check
demo_cov = tool_getInsuranceCoverageBlock("CBHI")
demo_near = tool_getNearbyHospitals(ALU_LAT, ALU_LON, MAX_RADIUS_KM, demo_cov)
demo_rank = tool_rankHospitalsByPriorityAndCost(demo_near["results"][:5], demo_cov)
print(f"Demo top pick: {demo_rank['rankedResults'][0]['result']['hospital']['name']}")
print(f"Est. copay: {demo_rank['rankedResults'][0]['estimatedCopayRwf']:,} RWF")

## 6 — Trajectory builder

Assembles a complete multi-turn conversation. Tool calls are deterministic; only the user query may be LLM-generated.

In [ ]:
SYSTEM_PROMPT = textwrap.dedent("""\
    You are Ranga, an offline hospital recommendation assistant for Rwanda.
    You MUST call tools in this order:
      1. getCurrentLocation
      2. getInsuranceCoverageBlock
      3. getNearbyHospitals OR searchHospitalsByCondition
      4. rankHospitalsByPriorityAndCost
    Never guess provider names, network status, or copay amounts.

    Safety rules:
    1. NEVER diagnose. 2. NEVER name drugs. 3. NEVER recommend out-of-network providers.
    4. ALWAYS call rankHospitalsByPriorityAndCost before stating a price.
    5. Mental health crisis → respond only: EMERGENCY_SOS
    """).strip()

CONDITION_KEYWORDS = re.compile(
    r'\b(dental|dent|teeth|tooth|eye|optic|vision|chest|pain|bone|emerg|accident|injur|therap|symptom)\b', re.I)

def _tool_call(name: str, arguments: dict) -> dict:
    return {"type": "function", "function": {"name": name, "arguments": arguments}}

def _tool_msg(call_id: str, name: str, result: dict) -> dict:
    return {"role": "tool", "tool_call_id": call_id, "name": name,
            "content": json.dumps(result, ensure_ascii=False)}

def _assistant_tool_turn(call_id: str, name: str, arguments: dict, result: dict) -> list[dict]:
    return [
        {"role": "assistant", "content": None, "tool_calls": [{"id": call_id, **_tool_call(name, arguments)}]},
        _tool_msg(call_id, name, result),
    ]

def build_final_response(rank_entry: dict, coverage: dict, service: str,
                         has_referral: bool, bypass: bool) -> str:
    h = rank_entry["result"]["hospital"]
    oop = rank_entry["estimatedCopayRwf"]
    dist = rank_entry["result"]["distanceKm"]
    in_net = rank_entry["result"]["isInNetwork"]
    lines = []
    if coverage["requiresReferral"] and not has_referral and not bypass:
        lines.append(f"Your {coverage['providerName']} plan requires a referral from your local health post.")
    elif bypass:
        lines.append("Your chronic visit history allows you to bypass the referral requirement.")
    if not in_net:
        lines.append(f"Warning: {h['name']} is out-of-network for your plan.")
    else:
        lines.append(f"{h['name']} ({dist:.1f} km away) covers {service} under {coverage['providerName']}.")
    lines.append(f"Estimated copayment: {oop:,} RWF ({coverage['copayPercent']:.0f}% of base fee).")
    lines.append(coverage["referralNotes"])
    lines.append("Bring your insurance card.")
    return " ".join(lines)

def use_condition_path(query: str, service: str) -> bool:
    return bool(CONDITION_KEYWORDS.search(query)) or service in ("Dental Care", "Optical Care", "Emergency Services", "Physical Therapy")

def build_tool_trajectory(user_query: str, scheme: str, service: str,
                          has_referral: bool, has_chronic: bool) -> tuple[list[dict], list[dict], str]:
    """Returns (messages, tool_log, pipeline_name)."""
    bypass = has_chronic
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_query}]
    tool_log, step = [], 0

    step += 1; cid = f"call_{step}"
    loc = execute_tool("getCurrentLocation", {})
    tool_log.append({"tool": "getCurrentLocation", "args": {}, "result": loc})
    messages.extend(_assistant_tool_turn(cid, "getCurrentLocation", {}, loc))

    step += 1; cid = f"call_{step}"
    cov_args = {"insurance": scheme}
    cov = execute_tool("getInsuranceCoverageBlock", cov_args)
    tool_log.append({"tool": "getInsuranceCoverageBlock", "args": cov_args, "result": cov})
    messages.extend(_assistant_tool_turn(cid, "getInsuranceCoverageBlock", cov_args, cov))

    step += 1; cid = f"call_{step}"
    if use_condition_path(user_query, service):
        hosp_name = "searchHospitalsByCondition"
        hosp_args = {"condition": service, "coverageBlock": cov, "lat": loc["lat"], "lng": loc["lng"]}
        pipeline = "condition"
    else:
        hosp_name = "getNearbyHospitals"
        hosp_args = {"lat": loc["lat"], "lng": loc["lng"], "radiusKm": MAX_RADIUS_KM, "coverageBlock": cov}
        pipeline = "nearby"
    hosp_res = execute_tool(hosp_name, hosp_args)
    tool_log.append({"tool": hosp_name, "args": {k: v for k, v in hosp_args.items() if k != "coverageBlock"}, "result_count": len(hosp_res["results"])})
    messages.extend(_assistant_tool_turn(cid, hosp_name, hosp_args, hosp_res))

    step += 1; cid = f"call_{step}"
    rank_args = {"hospitals": hosp_res["results"], "coverageBlock": cov}
    rank_res = execute_tool("rankHospitalsByPriorityAndCost", rank_args)
    tool_log.append({"tool": "rankHospitalsByPriorityAndCost", "args": {"hospital_count": len(hosp_res["results"])}, "top_score": rank_res["rankedResults"][0]["score"]})
    messages.extend(_assistant_tool_turn(cid, "rankHospitalsByPriorityAndCost", rank_args, rank_res))

    top = rank_res["rankedResults"][0]
    final = build_final_response(top, cov, service, has_referral, bypass)
    messages.append({"role": "assistant", "content": final})
    return messages, tool_log, pipeline

def pick_rule(service: str, scheme: str | None = None) -> dict | None:
    mask = rules_in_network["service_category"] == service
    if scheme: mask &= rules_in_network["scheme_name"] == scheme
    pool = rules_in_network[mask]
    return None if pool.empty else pool.sample(1).iloc[0].to_dict()

def make_training_record(messages, metadata) -> dict:
    return {"tools": RANGA_TOOLS, "messages": messages, "metadata": metadata}

## 7 — Generate SFT trajectories

### 7a — Template (300) — deterministic

In [ ]:
TRAIN_SCHEMES = ["CBHI", "RSSB", "Britam", "Old Mutual"]
QUERY_TEMPLATES = [
    "I use {scheme}. I need {service}. Which covered clinic near ALU should I go to?",
    "I use {scheme} and have a referral from my health post. I need {service}. Where should I go?",
    "I use {scheme}. I have {service} needs and a long history with this condition. Do I still need a referral?",
    "My {scheme} claim at {prior} was rejected. Find a covered {service} provider near ALU.",
]
PRIOR_FAILURE_NAMES = ["Legacy Hospital", "Legacy Clinics"]

template_pairs, attempts = [], 0
while len(template_pairs) < N_TEMPLATE and attempts < N_TEMPLATE * 5:
    attempts += 1
    scheme, service = random.choice(TRAIN_SCHEMES), random.choice(SERVICES)
    rule = pick_rule(service, scheme)
    if rule is None: continue
    has_referral, has_chronic = random.random() > 0.3, random.random() > 0.6
    prior_name = random.choice(PRIOR_FAILURE_NAMES) if random.random() < 0.2 else None
    if prior_name:
        query = QUERY_TEMPLATES[3].format(scheme=scheme, service=service.lower(), prior=prior_name)
    elif has_chronic:
        query = QUERY_TEMPLATES[2].format(scheme=scheme, service=service)
    elif has_referral:
        query = QUERY_TEMPLATES[1].format(scheme=scheme, service=service.lower())
    else:
        query = QUERY_TEMPLATES[0].format(scheme=scheme, service=service.lower())
    messages, tool_log, pipeline = build_tool_trajectory(query, scheme, service, has_referral, has_chronic)
    rank_msg = next(m for m in reversed(messages) if m.get("name") == "rankHospitalsByPriorityAndCost")
    top = json.loads(rank_msg["content"])["rankedResults"][0]
    template_pairs.append(make_training_record(messages, {
        "type": "template", "scheme": scheme, "service": service, "pipeline": pipeline,
        "tool_calls": len(tool_log), "oop_rwf": top["estimatedCopayRwf"],
        "top_provider": top["result"]["hospital"]["name"],
        "prior_failure": prior_name,
    }))
print(f"Generated {len(template_pairs)} template trajectories.")

### 7b — Custom (200) — LLM writes user query only

In [ ]:
from tqdm.notebook import tqdm

SAFETY_CHECK = re.compile(
    r'\b(diagnos\w+|prescri\w+|antibiotic|ibuprofen|paracetamol|amoxicillin|'
    r'metformin|medication|drug|tablet|mg |ml |injection|biopsy|surgery|'
    r'you have|you may have|this could be|sounds like a)\b', re.IGNORECASE)

def generate_user_query(ctx: dict, scheme: str, service: str, prior_failure: str | None) -> str | None:
    hint = f"Their last claim at {prior_failure} was rejected. " if prior_failure else ""
    prompt = textwrap.dedent(f"""\
        Write ONE short spoken sentence a Rwandan university student would say to a hospital routing app.
        Insurance: {scheme} | Service: {service} | Age: {ctx['age_band']}
        Context: {ctx['complaint_seed']}
        {hint}
        No hospital names. No prices. No diagnosis. No drug names. Query only.
        """).strip()
    raw = llm(prompt, max_tokens=80, temp=TEMPERATURE).strip().strip('"')
    return None if SAFETY_CHECK.search(raw) or len(raw) < 15 else raw

custom_pairs, ctx_pool = [], contexts.copy()
random.shuffle(ctx_pool)
pbar = tqdm(total=N_CUSTOM, desc="Custom trajectories")
idx = 0
while len(custom_pairs) < N_CUSTOM and idx < len(ctx_pool) * 4:
    ctx = ctx_pool[idx % len(ctx_pool)]; idx += 1
    scheme = random.choice(TRAIN_SCHEMES)
    service = ctx["service_category"]
    if pick_rule(service, scheme) is None: continue
    prior_name = random.choice(PRIOR_FAILURE_NAMES) if random.random() < 0.25 else None
    query = generate_user_query(ctx, scheme, service, prior_name)
    if query is None: continue
    has_referral = "referral" in query.lower()
    has_chronic = ctx["has_chronic"] or "history" in query.lower() or "long time" in query.lower()
    messages, tool_log, pipeline = build_tool_trajectory(query, scheme, service, has_referral, has_chronic)
    if SAFETY_CHECK.search(messages[-1]["content"]): continue
    rank_msg = next(m for m in reversed(messages) if m.get("name") == "rankHospitalsByPriorityAndCost")
    top = json.loads(rank_msg["content"])["rankedResults"][0]
    custom_pairs.append(make_training_record(messages, {
        "type": "custom", "source_row_id": ctx["source_row_id"],
        "scheme": scheme, "service": service, "pipeline": pipeline,
        "tool_calls": len(tool_log), "oop_rwf": top["estimatedCopayRwf"],
        "top_provider": top["result"]["hospital"]["name"], "prior_failure": prior_name,
    }))
    pbar.update(1)
pbar.close()
print(f"Generated {len(custom_pairs)} custom trajectories.")

### 7c — Save SFT file

In [ ]:
sft_pairs = template_pairs + custom_pairs
random.shuffle(sft_pairs)
with open(SFT_PATH, "w") as f:
    for rec in sft_pairs:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

sample = random.choice(sft_pairs)
print(f"Saved {len(sft_pairs)} records → {SFT_PATH.name}")
print(f"Pipeline: {sample['metadata']['pipeline']} | Tools called: {sample['metadata']['tool_calls']}")
for m in sample["messages"]:
    if m["role"] == "assistant" and m.get("tool_calls"):
        for tc in m["tool_calls"]:
            print(f"  → {tc['function']['name']}")
    elif m["role"] == "assistant" and m.get("content"):
        print(f"  ✓ {m['content'][:100]}…")

## 8 — Generate DPO pairs

Rejected trajectories teach incorrect tool-use patterns.

In [ ]:
DPO_DISTRIBUTION = {
    "skipped_tool_calls":  60,
    "out_of_network":      50,
    "wrong_tier":          40,
    "missing_pre_auth":    30,
    "hallucinated_cost":   20,
}
assert sum(DPO_DISTRIBUTION.values()) == N_DPO

def corrupt_trajectory(chosen: dict, flaw: str) -> dict | None:
    rejected = copy.deepcopy(chosen)
    msgs, final, meta = rejected["messages"], rejected["messages"][-1]["content"], chosen["metadata"]
    oop_match = re.search(r'(\d[\d,]+) RWF', final)
    oop = int(oop_match.group(1).replace(",", "")) if oop_match else 15_000

    if flaw == "skipped_tool_calls":
        rejected["messages"] = [msgs[0], msgs[1], {"role": "assistant", "content": final}]
    elif flaw == "out_of_network":
        prior = meta.get("prior_failure") or "Legacy Hospital"
        rejected["messages"][-1]["content"] = f"Go to {prior}. " + final
    elif flaw == "wrong_tier":
        rejected["messages"][-1]["content"] = "You can go directly without a referral. " + final
    elif flaw == "missing_pre_auth":
        rejected["messages"][-1]["content"] = re.sub(r'Obtain pre-authorisation.*?\.\s*', '', final, flags=re.I)
    elif flaw == "hallucinated_cost":
        rejected["messages"][-1]["content"] = re.sub(r'\d[\d,]+ RWF', f'{oop//2:,} RWF', final, count=1)
    else:
        return None
    rejected["metadata"] = {**meta, "flaw": flaw}
    return rejected

dpo_pairs, sft_pool = [], [p for p in sft_pairs if p["metadata"]["type"] == "template"]
random.shuffle(sft_pool)
pool_idx = 0
for flaw, target in DPO_DISTRIBUTION.items():
    collected = 0
    pbar = tqdm(total=target, desc=flaw)
    while collected < target and pool_idx < len(sft_pool) * 3:
        chosen_rec = sft_pool[pool_idx % len(sft_pool)]; pool_idx += 1
        rejected_rec = corrupt_trajectory(chosen_rec, flaw)
        if rejected_rec is None: continue
        dpo_pairs.append({"tools": RANGA_TOOLS, "chosen": chosen_rec, "rejected": rejected_rec,
                          "rejection_type": flaw, "metadata": chosen_rec["metadata"]})
        collected += 1; pbar.update(1)
    pbar.close()
print(f"Total DPO pairs: {len(dpo_pairs)}")

In [ ]:
with open(DPO_PATH, "w") as f:
    for pair in dpo_pairs:
        f.write(json.dumps(pair, ensure_ascii=False) + "\n")
from collections import Counter
print(f"Saved {len(dpo_pairs)} DPO pairs → {DPO_PATH.name}")
for k, v in Counter(p["rejection_type"] for p in dpo_pairs).most_common():
    print(f"  {k:<22} {v}")

## 9 — Generate eval queries (50)

In [ ]:
VOICE_TEMPLATES = [
    "I use {scheme}. I need {service}. Where should I go near ALU?",
    "My insurance is {scheme} and I have {service} needs. Which clinic is covered?",
    "I need {service} — I am on {scheme}. What is my cheapest covered option?",
]

eval_queries, attempts = [], 0
while len(eval_queries) < N_EVAL and attempts < N_EVAL * 10:
    attempts += 1
    scheme, service = random.choice(TRAIN_SCHEMES), random.choice(SERVICES)
    rule = pick_rule(service, scheme)
    if rule is None: continue
    query = random.choice(VOICE_TEMPLATES).format(scheme=scheme, service=service.lower())
    has_chronic = random.random() > 0.6
    if has_chronic: query += " I have been dealing with this for a long time."

    messages, tool_log, pipeline = build_tool_trajectory(query, scheme, service, False, has_chronic)
    expected_tools = [{"name": entry["tool"], "arguments": entry.get("args", {})} for entry in tool_log]
    rank_msg = next(m for m in reversed(messages) if m.get("name") == "rankHospitalsByPriorityAndCost")
    top = json.loads(rank_msg["content"])["rankedResults"][0]

    eval_queries.append({
        "query": query, "tools": RANGA_TOOLS,
        "expected_pipeline": pipeline,
        "expected_tool_calls": expected_tools,
        "expected_service": service, "expected_scheme": scheme,
        "expected_provider": top["result"]["hospital"]["id"],
        "expected_oop_rwf": top["estimatedCopayRwf"],
        "ground_truth_label": "correct_tool_trajectory",
    })

with open(EVAL_PATH, "w") as f:
    for q in eval_queries:
        f.write(json.dumps(q, ensure_ascii=False) + "\n")
print(f"Saved {len(eval_queries)} eval records → {EVAL_PATH.name}")

## 10 — Validation report

In [ ]:
REQUIRED_FINAL_TOOL = "rankHospitalsByPriorityAndCost"

def extract_tool_names(messages: list) -> list[str]:
    names = []
    for m in messages:
        if m.get("tool_calls"):
            names += [tc["function"]["name"] for tc in m["tool_calls"]]
    return names

def validate_trajectory(messages: list) -> dict:
    tools = extract_tool_names(messages)
    pipeline = "condition" if "searchHospitalsByCondition" in tools else "nearby"
    expected = TOOL_PIPELINE_CONDITION if pipeline == "condition" else TOOL_PIPELINE_NEARBY
    return {
        "tool_count": len(tools),
        "pipeline": pipeline,
        "has_rank": REQUIRED_FINAL_TOOL in tools,
        "correct_order": tools == expected,
    }

def count_clinical_violations(path: pathlib.Path) -> int:
    n = 0
    with open(path) as f:
        for line in f:
            rec = json.loads(line)
            msgs = rec.get("messages", rec.get("chosen", {}).get("messages", []))
            text = " ".join(m.get("content") or "" for m in msgs)
            if SAFETY_CHECK.search(text): n += 1
    return n

def line_count(p): return sum(1 for _ in open(p)) if p.exists() else 0

print("=" * 55)
print("VALIDATION REPORT")
print("=" * 55)
print(f"Tools ({len(RANGA_TOOLS)}): {', '.join(t['function']['name'] for t in RANGA_TOOLS)}")
print(f"Providers: {len(hospitals)} | In-network rules: {len(rules_in_network)}")
print()

order_ok = sum(1 for r in sft_pairs if validate_trajectory(r["messages"])["correct_order"])
rank_ok  = sum(1 for r in sft_pairs if validate_trajectory(r["messages"])["has_rank"])
print(f"SFT   lines={line_count(SFT_PATH)}  correct_pipeline_order={order_ok}/{len(sft_pairs)}")
print(f"      has_rank_call={rank_ok}/{len(sft_pairs)}  clinical_violations={count_clinical_violations(SFT_PATH)}")
print(f"DPO   lines={line_count(DPO_PATH)}")
print(f"Eval  lines={line_count(EVAL_PATH)}")
print()
avg_tools = sum(r["metadata"]["tool_calls"] for r in sft_pairs) / max(len(sft_pairs), 1)
nearby_n  = sum(1 for r in sft_pairs if r["metadata"].get("pipeline") == "nearby")
cond_n    = sum(1 for r in sft_pairs if r["metadata"].get("pipeline") == "condition")
print(f"Avg tool calls/record: {avg_tools:.1f} | nearby={nearby_n} condition={cond_n}")
print(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")